In [1]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt
import os
from rasterio.errors import RasterioIOError
import csv
import sys
if '/mnt/d/HLS Kelp Detection/tools' not in sys.path:
    sys.path.append('/mnt/d/HLS Kelp Detection/tools')
import kelp_tools_linux as kt
import data_tools as dt
import cupy as cp

In [2]:
tile = '10UCU'
location = 'Isla_vista_kelp'
cloud_cover_threshold = .25
save_mask = True
save_classification = True
path = '/mnt/d/HLS_Kelp/imagery/tiles'
items =['HLS.L30.T19QFV.2018215T144932.v2.0',
        'HLS.L30.T19QFV.2018167T144907.v2.0',
        'HLS.L30.T19QFV.2018199T144924.v2.0',
        'HLS.L30.T19QFV.2017340T145022.v2.0',
        'HLS.L30.T19QFV.2017244T145019.v2.0',
        'HLS.L30.T19QFV.2017276T145028.v2.0',
        'HLS.S30.T20PQC.2020364T143731.v2.0',
        'HLS.S30.T20PPC.2020027T144701.v2.0',
        'HLS.S30.T20PQC.2020354T143731.v2.0',
        'HLS.L30.T19QFV.2017132T144937.v2.0',
        'HLS.L30.T19QFV.2017228T145016.v2.0']

save_to = '/mnt/d/HLS_Kelp/random_forest/training_data/sargassum_unclassified'
if not os.path.isdir(save_to):
    os.mkdir(save_to)
show_dem = True
use_kmeans = False

In [ ]:
for item in items:
    tile = dt.extract_tile_id(item)
    tile_path = os.path.join(path,tile)
    img_path = os.path.join(path,tile, item)
    classified_path = os.path.join('/mnt/d/HLS_Kelp/processed imagery/tiles',tile,f"{item}.tif")
    dem_path = os.path.join(path,tile,'dem')
    img_files = kt.filter_and_sort_files(img_path,item)

    geotiff_path = os.path.join(img_path, img_files[0])

    land_mask = kt.create_land_mask(hls_path=geotiff_path, dem_path=dem_path, show_image=show_dem, as_numpy=True)

    cloud_land_mask, cloud_but_not_land_mask, percent_cloud_covered = kt.create_qa_mask(land_mask, img_path=img_path, as_numpy=True)
    dt.view_img(img_path)

    ##==========Create stacked np array, Apply landmask==========##
    img_bands = []
    crs = None
    transform = None
    try:
        for file in img_files:
            with rasterio.open(os.path.join(img_path, file)) as src:
                img_bands.append(np.where(land_mask, 0, np.asarray(src.read(1))))
                if(transform is None):
                    transform = src.transform
                    crs= src.crs
    except RasterioIOError as e:
        print(f"Error reading file {file} in granule {item}: {e}")
        sys.exit()

    img = np.stack(img_bands, axis=0)
    n_bands, height, width = img.shape
    img_normalized = kt.normalize_img(img,flatten=True, as_numpy=True)
    with rasterio.open(classified_path, 'r') as src:
        combined_mask = src.read(1)
    combined_mask = np.where(combined_mask == 4, 3, combined_mask)
    combined_mask = combined_mask + 1
    plt.figure(figsize=(20, 10))
    plt.imshow(np.array(combined_mask), cmap='Blues')
    plt.show()

    if save_classification:
        if not os.path.isdir (save_to):
            os.mkdir(save_to)
        classification_path = os.path.join(save_to,f'{item}_unclassified.tif') 
        height, width = combined_mask.shape
        reshaped_normalized_img = np.array(img_normalized)
        combined_img = np.dstack((reshaped_normalized_img,combined_mask))

        plt.figure(figsize=(6, 6))
        plt.imshow(combined_img[:,:,4], cmap='gray')
        plt.show()  

        num_bands = 7
        data_type = rasterio.uint8
        profile = {
            'driver': 'GTiff',
            'width': width,
            'height': height,
            'count': 7,  # one band
            'dtype': data_type,  # assuming binary mask, adjust dtype if needed
            'crs': crs,
            'transform': transform,
            'nodata': 0  # assuming no data is 0
        }
        # Write the land mask array to GeoTIFF
        with rasterio.open(classification_path, 'w', **profile) as dst:
            for i in range(num_bands):
                dst.write(combined_img[:,:,i].astype(data_type), i + 1)